In [1]:
import requests
import json
import pprint
import pandas as pd
import os
from urllib.parse import quote
import time
import asyncio
import aiohttp
from typing import Dict, List

# Đường dẫn

In [2]:
BASE_URL = 'https://api.open-meteo.com/v1/forecast'
AQ_URL   = 'https://air-quality-api.open-meteo.com/v1/air-quality'


# **TEST CHO TP HCM**
## **1. Weather**

In [3]:
r_wea = requests.get(BASE_URL, params={
    'latitude' : 10.7769,
    'longitude': 106.7009,
    'current' : 'temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation',
    'timezone': 'Asia/Ho_Chi_Minh',
}, timeout=30)

print('Status: ', r_wea.status_code)
pprint.pprint(r_wea.json())

Status:  200
{'current': {'interval': 900,
             'precipitation': 0.0,
             'relative_humidity_2m': 85,
             'temperature_2m': 27.1,
             'time': '2026-04-08T21:30',
             'wind_speed_10m': 10.9},
 'current_units': {'interval': 'seconds',
                   'precipitation': 'mm',
                   'relative_humidity_2m': '%',
                   'temperature_2m': '°C',
                   'time': 'iso8601',
                   'wind_speed_10m': 'km/h'},
 'elevation': 12.0,
 'generationtime_ms': 0.05924701690673828,
 'latitude': 10.75,
 'longitude': 106.75,
 'timezone': 'Asia/Ho_Chi_Minh',
 'timezone_abbreviation': 'GMT+7',
 'utc_offset_seconds': 25200}


## **2. AIR**

In [4]:
r_aq = requests.get(AQ_URL, params={
    'latitude': 10.7769,
    'longitude': 106.7009,
    'current': 'pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,ozone,uv_index,us_aqi',
    'timezone': 'Asia/Ho_Chi_Minh',
}, timeout=30)

print('Status: ', r_aq.status_code)
pprint.pprint(r_aq.json())

Status:  200
{'current': {'carbon_monoxide': 449.0,
             'interval': 3600,
             'nitrogen_dioxide': 28.4,
             'ozone': 57.0,
             'pm10': 25.7,
             'pm2_5': 24.2,
             'time': '2026-04-08T21:00',
             'us_aqi': 83,
             'uv_index': 0.0},
 'current_units': {'carbon_monoxide': 'μg/m³',
                   'interval': 'seconds',
                   'nitrogen_dioxide': 'μg/m³',
                   'ozone': 'μg/m³',
                   'pm10': 'μg/m³',
                   'pm2_5': 'μg/m³',
                   'time': 'iso8601',
                   'us_aqi': 'USAQI',
                   'uv_index': ''},
 'elevation': 12.0,
 'generationtime_ms': 0.2690553665161133,
 'latitude': 10.800003,
 'longitude': 106.70001,
 'timezone': 'Asia/Ho_Chi_Minh',
 'timezone_abbreviation': 'GMT+7',
 'utc_offset_seconds': 25200}


## **TEST TRÊN 3 TỈNH**

In [5]:
# Danh sách tỉnh thành Việt Nam
provinces = [
    {'code': '01', 'name': 'Hà Nội'},
    {'code': '02', 'name': 'Thành phố Hồ Chí Minh'},
    {'code': '03', 'name': 'Hải Phòng'},
    {'code': '04', 'name': 'Đà Nẵng'},
    {'code': '05', 'name': 'Hà Giang'},
    {'code': '06', 'name': 'Cao Bằng'},
    {'code': '07', 'name': 'Lai Châu'},
    {'code': '08', 'name': 'Lào Cai'},
    {'code': '09', 'name': 'Tuyên Quang'},
    {'code': '10', 'name': 'Lạng Sơn'},
    {'code': '11', 'name': 'Bắc Kạn'},
    {'code': '12', 'name': 'Thái Nguyên'},
    {'code': '13', 'name': 'Yên Bái'},
    {'code': '14', 'name': 'Sơn La'},
    {'code': '15', 'name': 'Phú Thọ'},
    {'code': '16', 'name': 'Vĩnh Phúc'},
    {'code': '17', 'name': 'Quảng Ninh'},
    {'code': '18', 'name': 'Bắc Giang'},
    {'code': '19', 'name': 'Bắc Ninh'},
    {'code': '21', 'name': 'Hải Dương'},
    {'code': '22', 'name': 'Hưng Yên'},
    {'code': '23', 'name': 'Hòa Bình'},
    {'code': '24', 'name': 'Hà Nam'},
    {'code': '25', 'name': 'Nam Định'},
    {'code': '26', 'name': 'Thái Bình'},
    {'code': '27', 'name': 'Ninh Bình'},
    {'code': '28', 'name': 'Thanh Hóa'},
    {'code': '29', 'name': 'Nghệ An'},
    {'code': '30', 'name': 'Hà Tĩnh'},
    {'code': '31', 'name': 'Quảng Bình'},
    {'code': '32', 'name': 'Quảng Trị'},
    {'code': '33', 'name': 'Thừa Thiên Huế'},
    {'code': '34', 'name': 'Quảng Nam'},
    {'code': '35', 'name': 'Quảng Ngãi'},
    {'code': '36', 'name': 'Kon Tum'},
    {'code': '37', 'name': 'Bình Định'},
    {'code': '38', 'name': 'Gia Lai'},
    {'code': '39', 'name': 'Phú Yên'},
    {'code': '40', 'name': 'Đắk Lắk'},
    {'code': '41', 'name': 'Khánh Hòa'},
    {'code': '42', 'name': 'Lâm Đồng'},
    {'code': '43', 'name': 'Bình Phước'},
    {'code': '44', 'name': 'Bình Dương'},
    {'code': '45', 'name': 'Ninh Thuận'},
    {'code': '46', 'name': 'Tây Ninh'},
    {'code': '47', 'name': 'Bình Thuận'},
    {'code': '48', 'name': 'Đồng Nai'},
    {'code': '49', 'name': 'Long An'},
    {'code': '50', 'name': 'Đồng Tháp'},
    {'code': '51', 'name': 'An Giang'},
    {'code': '52', 'name': 'Bà Rịa-Vũng Tàu'},
    {'code': '53', 'name': 'Tiền Giang'},
    {'code': '54', 'name': 'Kiên Giang'},
    {'code': '55', 'name': 'Cần Thơ'},
    {'code': '56', 'name': 'Bến Tre'},
    {'code': '57', 'name': 'Vĩnh Long'},
    {'code': '58', 'name': 'Trà Vinh'},
    {'code': '59', 'name': 'Sóc Trăng'},
    {'code': '60', 'name': 'Bạc Liêu'},
    {'code': '61', 'name': 'Cà Mau'},
    {'code': '62', 'name': 'Điện Biên'},
    {'code': '63', 'name': 'Đắk Nông'},
    {'code': '64', 'name': 'Hậu Giang'}
]


In [6]:
lats = [10.7769, 21.0285, 16.0472]  # HCM, HN, DN
lons = [106.7009, 105.8542, 108.2208]


r_batch = requests.get(BASE_URL, params={
    'latitude':  ','.join(map(str, lats)),
    'longitude': ','.join(map(str, lons)),
    'current': 'temperature_2m,wind_speed_10m',
    'timezone': 'Asia/Ho_Chi_Minh',
}, timeout=30)

data = r_batch.json()
print('Type:', type(data))       # list khi batch
print('Len:', len(data))          # 3
print('Keys: ',data[0].keys(), data[1].keys(), data[2].keys())
df_all = pd.DataFrame(data)

Type: <class 'list'>
Len: 3
Keys:  dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'current_units', 'current']) dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'location_id', 'current_units', 'current']) dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'location_id', 'current_units', 'current'])


In [7]:
df_all

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,current_units,current,location_id
0,10.75,106.750,0.037670,25200,Asia/Ho_Chi_Minh,GMT+7,12.0,"{'time': 'iso8601', 'interval': 'seconds', 'te...","{'time': '2026-04-08T21:30', 'interval': 900, ...",NaN
1,21.00,105.875,0.022769,25200,Asia/Ho_Chi_Minh,GMT+7,19.0,"{'time': 'iso8601', 'interval': 'seconds', 'te...","{'time': '2026-04-08T21:30', 'interval': 900, ...",1.0
2,16.00,108.125,0.022650,25200,Asia/Ho_Chi_Minh,GMT+7,7.0,"{'time': 'iso8601', 'interval': 'seconds', 'te...","{'time': '2026-04-08T21:30', 'interval': 900, ...",2.0


In [ ]:
# Mapbox token — KHÔNG hardcode trong code, dùng biến môi trường
# Lấy từ .env: MAPBOX_API_TOKEN
# Ví dụ: os.getenv("MAPBOX_API_TOKEN")
MAPBOX_TOKEN = os.getenv("MAPBOX_API_TOKEN", "YOUR_TOKEN_HERE")

### **Lấy tọa độ 63 tỉnh theo mã code ở trên**

In [9]:
def geocode_province(province_name, token, *, session=None, timeout=10, debug=False):
    """
    Lấy tọa độ tỉnh/thành từ Mapbox Geocoding API (KHÔNG in token).
    Trả về (lat, lon) hoặc (None, None)
    """
    # Chặn trường hợp truyền nhầm list/dict vào đây
    if not isinstance(province_name, str):
        if debug:
            print(f"[geocode_province] province_name phải là string, nhận {type(province_name)}")
        return None, None

    province_name = province_name.strip()
    if not province_name:
        return None, None

    # Encode query cho an toàn
    query = quote(f"{province_name}, Vietnam")
    url = f"https://api.mapbox.com/geocoding/v5/mapbox.places/{query}.json"

    params = {
        "access_token": token,
        "country": "VN",
        "types": "region,place",
        "limit": 1,
    }

    sess = session or requests

    try:
        r = sess.get(url, params=params, timeout=timeout)
        r.raise_for_status()
        data = r.json()

        features = data.get("features") or []
        if not features:
            if debug:
                print(f"Không tìm thấy tọa độ cho: {province_name}")
            return None, None

        lon, lat = features[0]["center"]
        return round(lat, 4), round(lon, 4)

    except requests.exceptions.HTTPError as e:
        status = getattr(e.response, "status_code", None)
        if debug:
            print(f"Lỗi HTTP khi lấy tọa độ cho {province_name} (status={status})")
        return None, None

    except requests.exceptions.RequestException as e:
        if debug:
            print(f"Lỗi mạng khi lấy tọa độ cho {province_name}: {type(e).__name__}")
        return None, None


def crawl_province(provinces, mapbox_token, output_dir, *, sleep_sec=0.1, debug=False):
    results = []

    print("Bắt đầu lấy tọa độ các tỉnh thành Việt Nam...")
    print("-" * 60)

    os.makedirs(output_dir, exist_ok=True)

    with requests.Session() as session:
        for i, province in enumerate(provinces, 1):
            name = province.get("name")
            code = province.get("code")

            print(f"[{i}/{len(provinces)}] Đang lấy: {name}: ", end="")

            lat, lon = geocode_province(
                name, mapbox_token, session=session, debug=debug
            )

            if lat is not None and lon is not None:
                results.append({
                    "MA_TINH": code,
                    "TEN_TINH": name,
                    "VI_DO": lat,
                    "KINH_DO": lon,
                })
                print(f"({lat}, {lon})")
            else:
                print("Thất bại")

            time.sleep(sleep_sec)

    print("-" * 60)
    print(f"Hoàn thành! Đã lấy được {len(results)}/{len(provinces)} tỉnh thành")

    df = pd.DataFrame(results)
    output_file = os.path.join(output_dir, "province.csv")
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Ghi vào file .csv thành công: {output_file}")

    return df



In [10]:
# Gọi hàm crawl_province để lấy tọa độ cho toàn bộ danh sách
df_provinces = crawl_province(provinces, MAPBOX_TOKEN, 'data_output', debug=True)

# Hiển thị 5 dòng đầu tiên
display(df_provinces.head())

Bắt đầu lấy tọa độ các tỉnh thành Việt Nam...
------------------------------------------------------------
[1/63] Đang lấy: Hà Nội: (21.0283, 105.854)
[2/63] Đang lấy: Thành phố Hồ Chí Minh: (10.7755, 106.7021)
[3/63] Đang lấy: Hải Phòng: (20.8623, 106.6799)
[4/63] Đang lấy: Đà Nẵng: (16.068, 108.212)
[5/63] Đang lấy: Hà Giang: (22.8279, 104.9823)
[6/63] Đang lấy: Cao Bằng: (22.6761, 106.2016)
[7/63] Đang lấy: Lai Châu: (22.3862, 103.4702)
[8/63] Đang lấy: Lào Cai: (22.4962, 103.968)
[9/63] Đang lấy: Tuyên Quang: (21.8212, 105.1833)
[10/63] Đang lấy: Lạng Sơn: (21.8511, 106.7622)
[11/63] Đang lấy: Bắc Kạn: (22.1398, 105.832)
[12/63] Đang lấy: Thái Nguyên: (21.5954, 105.8387)
[13/63] Đang lấy: Yên Bái: (21.7049, 104.8791)
[14/63] Đang lấy: Sơn La: (21.327, 103.9144)
[15/63] Đang lấy: Phú Thọ: (21.3135, 105.3946)
[16/63] Đang lấy: Vĩnh Phúc: (21.3079, 105.5965)
[17/63] Đang lấy: Quảng Ninh: (20.9489, 107.1035)
[18/63] Đang lấy: Bắc Giang: (21.2804, 106.1985)
[19/63] Đang lấy: Bắc Ninh: (

,MA_TINH,TEN_TINH,VI_DO,KINH_DO
0,01,Hà Nội,21.0283,105.8540
1,02,Thành phố Hồ Chí Minh,10.7755,106.7021
2,03,Hải Phòng,20.8623,106.6799
3,04,Đà Nẵng,16.0680,108.2120
4,05,Hà Giang,22.8279,104.9823


In [11]:

df_provinces = df_provinces.drop_duplicates(subset=['MA_TINH', 'TEN_TINH'], keep='first')

extra_data = [
    {'MA_TINH': '21', 'TEN_TINH': 'Hải Dương', 'VI_DO': 20.9411, 'KINH_DO': 106.3330},
    {'MA_TINH': '59', 'TEN_TINH': 'Sóc Trăng', 'VI_DO': 9.6025, 'KINH_DO': 105.9731}
]

for item in extra_data:
    if item['TEN_TINH'] in df_provinces['TEN_TINH'].values:
        df_provinces.loc[df_provinces['TEN_TINH'] == item['TEN_TINH'], ['VI_DO', 'KINH_DO']] = [item['VI_DO'], item['KINH_DO']]
    else:
        df_provinces = pd.concat([df_provinces, pd.DataFrame([item])], ignore_index=True)
print(f"Tổng số tỉnh thành hiện tại: {len(df_provinces)}")
display(df_provinces.tail())

Tổng số tỉnh thành hiện tại: 63


,MA_TINH,TEN_TINH,VI_DO,KINH_DO
58,62,Điện Biên,21.3924,103.0160
59,63,Đắk Nông,12.0006,107.6960
60,64,Hậu Giang,9.7832,105.4670
61,21,Hải Dương,20.9411,106.3330
62,59,Sóc Trăng,9.6025,105.9731


## **Thực hiện cào trên 63 tỉnh**

In [12]:
import logging

# Bật logger để thấy thông báo debug
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

WEATHER_URL = 'https://api.open-meteo.com/v1/forecast'
AQ_URL      = 'https://air-quality-api.open-meteo.com/v1/air-quality'

WEATHER_VARS = 'temperature_2m,wind_speed_10m,relative_humidity_2m,precipitation'
AQ_VARS      = 'pm10,pm2_5,nitrogen_dioxide,ozone,uv_index,us_aqi'

async def fetch_batch(session: aiohttp.ClientSession, url: str, params: Dict) -> List[Dict]:
    '''Fetch với retry 3 lần, exponential backoff'''
    for attempt in range(3):
        try:
            async with session.get(url, params=params, timeout=30) as resp:
                resp.raise_for_status()   # Raise nếu status 4xx/5xx
                data = await resp.json()
                # Open-Meteo trả về mảng (list) khi query nhiều tọa độ
                return data if isinstance(data, list) else [data]
        except aiohttp.ClientError as e:
            wait = 2 ** attempt  # 1s, 2s, 4s
            logger.warning(f'Attempt {attempt+1} failed: {e}. Retry in {wait}s')
            if attempt < 2:
                await asyncio.sleep(wait)
            else:
                logger.error(f'All 3 attempts failed for {url}')
                raise

async def collect_all_provinces(df: pd.DataFrame) -> pd.DataFrame:
    '''Fetch weather + AQ cho toàn bộ DataFrame trong 2 request song song'''

    lats = df['VI_DO'].tolist()
    lons = df['KINH_DO'].tolist()

    params = {
        'latitude':  ','.join(map(str, lats)),
        'longitude': ','.join(map(str, lons)),
        'timezone':  'Asia/Ho_Chi_Minh',
        'forecast_days': 1,
    }

    async with aiohttp.ClientSession() as session:
        weather_task = fetch_batch(session, WEATHER_URL, {**params, 'current': WEATHER_VARS})
        aq_task      = fetch_batch(session, AQ_URL, {**params, 'current': AQ_VARS})

        weather_list, aq_list = await asyncio.gather(weather_task, aq_task)

    records = []
    for row, w, aq in zip(df.to_dict('records'), weather_list, aq_list):
        record = {
            'MA_TINH':     row['MA_TINH'],
            'TEN_TINH':    row['TEN_TINH'],
            'VI_DO':       row['VI_DO'],
            'KINH_DO':     row['KINH_DO'],
            'time':        w['current'].get('time'),
            'temperature': w['current'].get('temperature_2m'),
            'humidity':    w['current'].get('relative_humidity_2m'),
            'wind_speed':  w['current'].get('wind_speed_10m'),
            'precipitation': w['current'].get('precipitation'),

            'pm2_5':       aq['current'].get('pm2_5'),
            'pm10':        aq['current'].get('pm10'),
            'aqi':         aq['current'].get('us_aqi'),
            'no2':         aq['current'].get('nitrogen_dioxide'),
            'ozone':       aq['current'].get('ozone'),
            'uv_index':    aq['current'].get('uv_index'),
            'raw_json':    {'weather': w, 'air_quality': aq},
        }
        records.append(record)

    logger.info(f'Collected {len(records)} province records')

    # Trả về đối tượng DataFrame để dễ bề phân tích tiếp ở Jupyter
    return pd.DataFrame(records)

# =========================================================
# CHẠY VÀ XUẤT OUTPUT (trong Cell mới)
# =========================================================
df_final = await collect_all_provinces(df_provinces)

# Lưu CSV để làm Dashboard hoặc EDA
df_final.to_csv('data_output/provinces_weather_aqi_meteo.csv', index=False, encoding='utf-8-sig')

display(df_final.head(10))


,MA_TINH,TEN_TINH,VI_DO,KINH_DO,time,temperature,humidity,wind_speed,precipitation,pm2_5,pm10,aqi,no2,ozone,uv_index,raw_json
0,01,Hà Nội,21.0283,105.8540,2026-04-08T21:30,28.3,79,13.8,0.0,48.1,49.6,205,30.1,98.0,0.0,"{'weather': {'latitude': 21.0, 'longitude': 10..."
1,02,Thành phố Hồ Chí Minh,10.7755,106.7021,2026-04-08T21:30,27.1,85,10.9,0.0,24.2,25.7,83,28.4,57.0,0.0,"{'weather': {'latitude': 10.75, 'longitude': 1..."
2,03,Hải Phòng,20.8623,106.6799,2026-04-08T21:30,26.3,87,7.4,0.0,25.1,28.7,113,8.1,124.0,0.0,"{'weather': {'latitude': 20.875, 'longitude': ..."
3,04,Đà Nẵng,16.0680,108.2120,2026-04-08T21:30,25.8,82,5.2,0.0,21.2,25.2,212,8.5,113.0,0.0,"{'weather': {'latitude': 16.125, 'longitude': ..."
4,05,Hà Giang,22.8279,104.9823,2026-04-08T21:30,25.2,80,5.8,0.0,53.6,59.1,151,10.8,128.0,0.0,"{'weather': {'latitude': 22.875, 'longitude': ..."
5,06,Cao Bằng,22.6761,106.2016,2026-04-08T21:30,27.5,84,3.8,0.0,65.0,68.2,200,8.9,175.0,0.0,"{'weather': {'latitude': 22.625, 'longitude': ..."
6,07,Lai Châu,22.3862,103.4702,2026-04-08T21:30,27.3,57,2.6,0.0,45.2,49.7,150,15.3,98.0,0.0,"{'weather': {'latitude': 22.375, 'longitude': ..."
7,08,Lào Cai,22.4962,103.9680,2026-04-08T21:30,30.2,53,2.5,0.0,48.3,52.4,151,21.9,80.0,0.0,"{'weather': {'latitude': 22.5, 'longitude': 10..."
8,09,Tuyên Quang,21.8212,105.1833,2026-04-08T21:30,31.2,81,7.0,0.0,83.1,85.9,188,14.4,159.0,0.0,"{'weather': {'latitude': 21.875, 'longitude': ..."
9,10,Lạng Sơn,21.8511,106.7622,2026-04-08T21:30,26.8,69,4.9,0.0,32.5,34.5,193,14.2,117.0,0.0,"{'weather': {'latitude': 21.875, 'longitude': ..."


In [17]:
import pprint

# Lấy 5 dòng đầu tiên để tránh in quá nhiều dữ liệu
for index, row in df_final.head(5).iterrows():
    print(f"--- Dữ liệu raw_json cho tỉnh {row['TEN_TINH']} ---")
    pprint.pprint(row['raw_json'])
    print("\n")

--- Dữ liệu raw_json cho tỉnh Hà Nội ---
{'air_quality': {'current': {'interval': 3600,
                             'nitrogen_dioxide': 30.1,
                             'ozone': 98.0,
                             'pm10': 49.6,
                             'pm2_5': 48.1,
                             'time': '2026-04-08T21:00',
                             'us_aqi': 205,
                             'uv_index': 0.0},
                 'current_units': {'interval': 'seconds',
                                   'nitrogen_dioxide': 'μg/m³',
                                   'ozone': 'μg/m³',
                                   'pm10': 'μg/m³',
                                   'pm2_5': 'μg/m³',
                                   'time': 'iso8601',
                                   'us_aqi': 'USAQI',
                                   'uv_index': ''},
                 'elevation': 11.0,
                 'generationtime_ms': 0.21791458129882812,
                 'latitude': 21.0,
        

# Kiểm tra chất lượng data

In [13]:
# In ra tổng số lượng giá trị null trên mỗi cột
print("Số lượng giá trị null ở mỗi cột:")
display(df_final.isnull().sum())

# Xác định tổng số giá trị null trên toàn bộ DataFrame
total_null = df_final.isnull().sum().sum()
print(f"\nTổng số ô dữ liệu bị trống: {total_null}")

# Lọc và hiển thị riêng các dòng có chứa dữ liệu bị thiếu (nếu có)
null_rows = df_final[df_final.isnull().any(axis=1)]
if not null_rows.empty:
    print(f"\nCó {len(null_rows)} dòng chứa giá trị null. Chi tiết các dòng đó:")
    display(null_rows)
else:
    print("\nDữ liệu của bạn rất sạch và không có biến null nào.")


Số lượng giá trị null ở mỗi cột:


,0
MA_TINH,0
TEN_TINH,0
VI_DO,0
KINH_DO,0
time,0
temperature,0
humidity,0
wind_speed,0
precipitation,0
pm2_5,0



Tổng số ô dữ liệu bị trống: 0

Dữ liệu của bạn rất sạch và không có biến null nào.
